# Imports

In [0]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql.functions import regexp_extract_all, regexp_replace,col, lit, size,  explode,split, element_at,col,udf,length,make_date,expr,weekofyear,collect_list,explode,max,min,lag,log,avg,when,abs,count,std,stddev
from pyspark.ml.feature import StringIndexer
from pyspark.sql.types import StringType
import json
from pyspark.sql import Window
import pandas as pd
import datetime

In [0]:
spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.getOrCreate()

# Data read


In [0]:
feature_store_technical  = spark.read.format('delta').load("/Volumes/market-mood/features/feature_store/") 
tweets = spark.read.format('delta').load('/Volumes/market-mood/processed/sentiment_scores')

In [0]:
tweets = tweets.drop('year','week')
tweets = tweets.join(feature_store_technical , on=['ticker','date'],how='left')

In [0]:
tweets.select('sentiment').show(5)

In [0]:
# indexer = StringIndexer(inputCol="sentiment", outputCol="sentiment_index")
# tweets = indexer.fit(tweets).transform(tweets)
# tweets.select('sentiment_index').show(5)
tweets = tweets.withColumn('sentiment_score',when(col('sentiment') == 'Bearish',-1).when(col('sentiment') == 'Bullish',1).otherwise(0))
tweets.select('sentiment_score').show(5)

In [0]:
tweets.printSchema()

In [0]:

tweets = tweets.select(
    "ticker", "date", "year", "week",
    "close", "volume",
    "log_return", "rsi", "macd", "bb_position",
    "volatility", "vol_ratio",
    "sentiment_score",
    "target"
)

tweets.printSchema()
tweets.count()

# Load

In [0]:
(tweets
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("ticker", "year")
    .save("/Volumes/market-mood/features/sentiment_store"))